# Metadata Filters and Access Control
Companion notebook for the **Metadata Filters and Access Control** section of Chapter 10.

Filter as early as you can, and verify again before generation. In enterprise RAG this is a safety control, not just a relevance tweak -- skip it and you may return the right-looking chunk from the wrong tenant.

A small stand-in for the `user` and `query_meta` objects the book's `build_filters` reads from:

In [1]:
from dataclasses import dataclass, field


@dataclass
class User:
    tenant_id: str
    groups: list[str] = field(default_factory=list)

This is the book's `build_filters`, unchanged:

In [2]:
def build_filters(user, query_meta):
    return {
        "tenant_id": user.tenant_id,
        "doc_type": query_meta.get("doc_type"),
        "source": query_meta.get("source"),
        "created_at": {"$gte": query_meta.get("from_date")},
        "allowed_groups": {"$in": user.groups},
    }

In [3]:
acme_support = User(tenant_id="acme", groups=["support", "manager"])
filters = build_filters(acme_support, {"doc_type": "policy", "source": "kb", "from_date": "2025-01-01"})
print(filters)

{'tenant_id': 'acme', 'doc_type': 'policy', 'source': 'kb', 'created_at': {'$gte': '2025-01-01'}, 'allowed_groups': {'$in': ['support', 'manager']}}


Applying the filter -- **pre-retrieval** to narrow the search space, and a **post-retrieval** check as a second line of defense, since permissions sometimes come from a separate system:

In [4]:
records = [
    {"chunk_id": "c1", "tenant_id": "acme", "doc_type": "policy", "created_at": "2026-01-15", "allowed_groups": ["support"]},
    {"chunk_id": "c2", "tenant_id": "globex", "doc_type": "policy", "created_at": "2026-02-01", "allowed_groups": ["support"]},
    {"chunk_id": "c3", "tenant_id": "acme", "doc_type": "runbook", "created_at": "2026-01-20", "allowed_groups": ["engineering"]},
    {"chunk_id": "c4", "tenant_id": "acme", "doc_type": "policy", "created_at": "2024-06-01", "allowed_groups": ["support"]},
]


def apply_filters(records: list[dict], filters: dict) -> list[dict]:
    """Pre-retrieval narrowing: tenant, doc_type, and a date floor."""
    return [
        r for r in records
        if r["tenant_id"] == filters["tenant_id"]
        and (filters["doc_type"] is None or r["doc_type"] == filters["doc_type"])
        and r["created_at"] >= filters["created_at"]["$gte"]
    ]


def verify_access(records: list[dict], user) -> list[dict]:
    """Post-retrieval check: confirm the user actually has an allowed group on each hit."""
    return [r for r in records if set(r["allowed_groups"]) & set(user.groups)]


narrowed = apply_filters(records, filters)
print("after pre-retrieval filter:", [r["chunk_id"] for r in narrowed])

verified = verify_access(narrowed, acme_support)
print("after post-retrieval access check:", [r["chunk_id"] for r in verified])

after pre-retrieval filter: ['c1']
after post-retrieval access check: ['c1']


`c2` (wrong tenant), `c3` (wrong doc type, and would fail the access check even if it passed filtering -- `acme_support` isn't in `engineering`), and `c4` (too old) are all correctly excluded, leaving only `c1`. Keep this logic in its own LangGraph node instead of hoping a prompt instruction enforces it -- that's what makes it visible, testable, and auditable.